# nb01c - Clean: provider counts for the access-gap view

Boils the 1.5 GB Medi-Cal Managed Care Provider Listing down to two small, join-ready summary tables:

1. `providers_by_county.csv` - distinct providers (by NPI) in each county, all plans, plus a primary-care-only count. Joins to county enrollment to give **providers per member** (network adequacy), the statewide access map.
2. `la_care_providers_by_type.csv` - distinct L.A. Care providers in Los Angeles by provider type, for the LA specialty view.

**Read in chunks.** The raw file has about 5 million rows, so it is read 600,000 rows at a time and only the needed columns are kept, to stay within memory.

**Counting rule.** A provider (one NPI) can appear many times, once per taxonomy, location, or plan. We count **distinct NPI**, so a provider is counted once per county.

**Delegation caveat.** Providers are counted under the plan they are listed with. L.A. Care delegates members to plan partners (Anthem, Blue Shield Promise, Molina), whose providers are listed under the partner's name, so the L.A. Care direct count understates its true accessible network. The statewide county map is unaffected, because it counts all providers in a county regardless of plan.

In [1]:
from pathlib import Path
from collections import defaultdict
import pandas as pd

DATA = Path('..') / 'data'
raw_files = sorted((DATA / 'raw').glob('mc_provider_listing_raw_*.csv'))
RAW = raw_files[-1]
print('reading', RAW.name, f'({RAW.stat().st_size/1_048_576:.0f} MB)')

USECOLS = ['ManagedCarePlan', 'County', 'MCNAProviderType', 'PrimaryCare', 'NPI']
PCP_TRUE = {'True', '1', 'Y', 'Yes', 'TRUE'}

county_all = defaultdict(set)   # county -> distinct NPI (all plans)
county_pcp = defaultdict(set)   # county -> distinct NPI flagged primary care
lac_type   = defaultdict(set)   # provider type -> distinct NPI (L.A. Care, LA only)

rows = 0
for ch in pd.read_csv(RAW, usecols=USECOLS, dtype=str, chunksize=600_000, low_memory=False):
    rows += len(ch)
    ch = ch.dropna(subset=['County', 'NPI'])
    ch['County'] = ch['County'].str.strip()
    for cty, npi, pc in zip(ch.County, ch.NPI, ch.PrimaryCare.fillna('')):
        county_all[cty].add(npi)
        if str(pc).strip() in PCP_TRUE:
            county_pcp[cty].add(npi)
    la = ch[(ch.County.str.lower() == 'los angeles') & (ch.ManagedCarePlan == 'L.A. Care Health Plan')]
    for t, npi in zip(la.MCNAProviderType.fillna('Unknown'), la.NPI):
        lac_type[t].add(npi)
print('rows scanned:', rows)

reading mc_provider_listing_raw_2026-07-22.csv (1511 MB)
rows scanned: 5043168


## 1. Providers by county (for the statewide access map)

In [2]:
county = (pd.DataFrame(
        [{'County': c, 'Providers_All': len(county_all[c]), 'Providers_PCP': len(county_pcp[c])}
         for c in county_all if isinstance(c, str) and c])
    .sort_values('Providers_All', ascending=False))
county.to_csv(DATA / 'providers_by_county.csv', index=False)
print('wrote providers_by_county.csv', county.shape)
print(county.head(8).to_string(index=False))

wrote providers_by_county.csv (58, 3)
       County  Providers_All  Providers_PCP
  Los Angeles          46237           5206
  Santa Clara          22153           1503
    San Diego          17377           2383
       Orange          15682           1418
   Sacramento          14864           1173
San Francisco          13632           1034
      Alameda          13519           1279
    Riverside          12737           1324


## 2. L.A. Care providers by type (for the LA specialty view)

In [3]:
by_type = (pd.DataFrame([{'Provider Type': t, 'Providers': len(s)} for t, s in lac_type.items()])
           .sort_values('Providers', ascending=False))
by_type.to_csv(DATA / 'la_care_providers_by_type.csv', index=False)
print('wrote la_care_providers_by_type.csv', by_type.shape)
print(by_type.head(10).to_string(index=False))

wrote la_care_providers_by_type.csv (29, 2)
                       Provider Type  Providers
                               Other       2850
                  Adult Primary Care       1125
                  Nurse Practitioner       1067
              Pediatric Primary Care        938
                 Physician Assistant        524
                     General Surgery        518
             Obstetrics & Gynecology        418
Cardiology/Interventional Cardiology        382
                       Ophthalmology        368
                  Orthopedic Surgery        300


## 3. QA - the access ratios the map will show

Providers per 1,000 members = distinct providers in a county divided by that county's Medi-Cal managed care members. Lower means a thinner network relative to need. This is computed in Tableau by joining `providers_by_county` to the county enrollment; here it is just a sanity check.

In [4]:
enr = pd.read_csv(DATA / 'ca_county_enrollment_clean.csv')
chk = county.merge(enr, on='County')   # VALIDATION ONLY; the real join is done in Tableau
chk['Providers per 1,000 members'] = (chk.Providers_All / chk['Medi-Cal MC Enrollees'] * 1000).round(1)
chk = chk.sort_values('Medi-Cal MC Enrollees', ascending=False)
print(chk.head(10)[['County', 'Medi-Cal MC Enrollees', 'Providers_All', 'Providers per 1,000 members']]
      .to_string(index=False))
print('\nlowest providers per 1,000 members (potential access gaps):')
print(chk.nsmallest(6, 'Providers per 1,000 members')[['County', 'Providers per 1,000 members']].to_string(index=False))

        County  Medi-Cal MC Enrollees  Providers_All  Providers per 1,000 members
   Los Angeles              3521674.0          46237                         13.1
San Bernardino               866361.0          12046                         13.9
     Riverside               862488.0          12737                         14.8
        Orange               855944.0          15682                         18.3
     San Diego               821390.0          17377                         21.2
    Sacramento               560275.0          14864                         26.5
        Fresno               473382.0           5760                         12.2
          Kern               448377.0           4113                          9.2
       Alameda               442000.0          13519                         30.6
   Santa Clara               397829.0          22153                         55.7

lowest providers per 1,000 members (potential access gaps):
  County  Providers per 1,000 members